# Zenodo oral-cavity dataset -- batch segmentation with your trained model

This notebook loads your trained Keras model (`stage3_cbam_sgd_final.keras`) and
runs it over **every image** in the Zenodo dataset (`Images/`, 3,000 files --
confirmed to match `Categories/`, `Imagewise_Data.csv` and `Annotation.json`,
all 3,000).

For each input image `X.jpg` it writes a predicted mask to `X.png` in a separate
output folder. **Nothing in your original `Images/` or `Categories/` folders is
ever touched, moved, renamed or deleted.**

### About the model file
Your upload is a native Keras 3 `.keras` archive (a zip of `config.json` +
`metadata.json` + `model.weights.h5` -- that's the format's actual internal
structure, so it loads straight through `keras.models.load_model()`, no manual
weight surgery needed). It uses one custom layer, `CBAMBottleneck` (channel +
spatial attention), which this notebook reimplements from the weight shapes
found in the file (`Dense 256->32`, `Dense 32->256`, `Conv2D` 7x7 over 2
channels -- a standard CBAM block) so `load_model` can resolve it.

The file also stores a compile config referencing a custom loss function
(`combined_loss`) that isn't included in the archive -- irrelevant for
inference, so this notebook loads with `compile=False` to skip it. (Verified
end-to-end already: loads cleanly, weights are non-random/trained-looking,
and a forward pass on a dummy image produces a sane probability map.)

### Running this locally
This version runs on your own machine (not Kaggle) -- **Section 1 asks you for
the paths interactively** (`input()` prompts) instead of hardcoded Kaggle
paths:
1. Run Section 1 and, when prompted, paste in: the path to your `.keras`
   model file, the path to your Zenodo `Images` folder, optionally the path
   to `Imagewise_Data.csv` (press Enter to skip it -- only needed for the
   coverage check and the per-class PPT figures), and the folder you want
   results written to (it's created for you if it doesn't exist).
2. Check `PREPROCESS_RESCALE` and `MASK_THRESHOLD`, set a bit further down in
   the same Section 1 cell -- rescale to `[0,1]` and threshold at `0.5` is my
   best assumption from the sigmoid output layer. If your training pipeline
   normalized differently (e.g. ImageNet mean/std), change
   `PREPROCESS_RESCALE` / `PREPROCESS_MEAN` / `PREPROCESS_STD` accordingly, or
   predictions will look wrong even though the model loaded fine. Section 3's
   sanity-check image is there so you can catch this before running the full
   3,000-image batch.
3. You'll need `tensorflow`/`keras`, `opencv-python`, `pandas`, and
   `matplotlib` installed locally (`pip install tensorflow opencv-python
   pandas matplotlib`).


In [ ]:
# ============================================================
# 1. CONFIG -- EDIT THESE PATHS FOR YOUR KAGGLE / LOCAL ENVIRONMENT
# ============================================================
from pathlib import Path


def ask_path(prompt, must_exist=True, is_file=False):
    while True:
        raw = input(prompt).strip().strip('"').strip("'")
        if not raw:
            print("  -> can't be empty, try again.")
            continue
        p = Path(raw).expanduser()
        if must_exist and not p.exists():
            print(f"  -> {p} doesn't exist, try again.")
            continue
        if must_exist and is_file and not p.is_file():
            print(f"  -> {p} exists but isn't a file, try again.")
            continue
        if must_exist and not is_file and not p.is_dir():
            print(f"  -> {p} exists but isn't a folder, try again.")
            continue
        return p


# --- Model ---
MODEL_PATH = ask_path("Path to your .keras model file (e.g. stage3_cbam_sgd_final.keras): ",
                       must_exist=True, is_file=True)

# --- Dataset ---
IMAGES_DIR = ask_path("Path to the Zenodo 'Images' folder (the 3,000 .jpg files): ",
                       must_exist=True, is_file=False)

# Optional -- only used for the coverage check (Section 4) and the per-class figures
# (Section 5). Leave blank to skip both.
_csv_raw = input("Path to Imagewise_Data.csv (optional, press Enter to skip): ").strip().strip('"').strip("'")
IMAGEWISE_CSV = Path(_csv_raw).expanduser() if _csv_raw else None

# --- Output ---
_base_out = ask_path("Output folder for results (created if it doesn't exist): ", must_exist=False)
_base_out.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR       = _base_out / "predicted_masks"        # binary B/W masks, same base filename as input
OVERLAY_DIR      = _base_out / "predicted_overlays"     # mask painted red on the original (visual QA only)
SEGMENTED_DIR    = _base_out / "segmented_oral_cavity"          # <-- THE ACTUAL SEGMENTED IMAGES YOU WANT:
                                                                  #     full-size original photo, everything
                                                                  #     outside the oral cavity blacked out
SEGMENTED_CROP_DIR = _base_out / "segmented_oral_cavity_cropped"  # same, but cropped tight to the region's
                                                                    # bounding box (smaller, zoomed in)

SAVE_OVERLAYS   = True     # set False to skip overlay images and save disk/time
SAVE_PROB_MAPS  = False    # if True, also saves raw float32 probability maps (.npy) -- large, off by default

# --- Model I/O ---
IMG_SIZE = 256                 # model's expected input resolution
MASK_THRESHOLD = 0.5           # sigmoid output -> binary mask cutoff
# Preprocessing assumption (see markdown above) -- adjust if your training used something else.
PREPROCESS_RESCALE = 1.0 / 255.0
PREPROCESS_MEAN = None         # e.g. [0.485, 0.456, 0.406] if ImageNet-normalized; None = skip
PREPROCESS_STD  = None         # e.g. [0.229, 0.224, 0.225]

BATCH_SIZE = 16

for d in [OUTPUT_DIR, OVERLAY_DIR, SEGMENTED_DIR, SEGMENTED_CROP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print()
print("Model path  :", MODEL_PATH)
print("Images dir  :", IMAGES_DIR)
print("CSV path    :", IMAGEWISE_CSV if IMAGEWISE_CSV else "(skipped)")
print("Masks              -> ", OUTPUT_DIR)
print("Overlays           -> ", OVERLAY_DIR)
print("Segmented images   -> ", SEGMENTED_DIR, "  <-- this is what you want")
print("Segmented (cropped)-> ", SEGMENTED_CROP_DIR)


In [ ]:
# ============================================================
# 2. LOAD THE MODEL
#    Reimplements the one custom layer (CBAMBottleneck) from the weight
#    shapes actually stored in the file, then loads the whole model in one
#    call -- since it's a self-consistent .keras archive there's no
#    name-matching risk like there would be with separately exported
#    config.json + weights.h5 files.
# ============================================================
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__, " Keras:", keras.__version__)


@keras.saving.register_keras_serializable(package="OralCancer")
class CBAMBottleneck(keras.layers.Layer):
    # Convolutional Block Attention Module (channel + spatial attention)
    def __init__(self, reduction_ratio=8, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.spatial_kernel = spatial_kernel

    def build(self, input_shape):
        channels = int(input_shape[-1])
        reduced = max(1, channels // self.reduction_ratio)
        self.channel_dense_1 = keras.layers.Dense(reduced, activation="relu", name="channel_dense_1")
        self.channel_dense_2 = keras.layers.Dense(channels, activation=None, name="channel_dense_2")
        self.spatial_conv = keras.layers.Conv2D(
            1, self.spatial_kernel, padding="same", activation=None, use_bias=True, name="spatial_conv")
        self.channel_dense_1.build((None, channels))
        self.channel_dense_2.build((None, reduced))
        self.spatial_conv.build((None, input_shape[1], input_shape[2], 2))
        super().build(input_shape)

    def call(self, x):
        avg_pool = keras.layers.GlobalAveragePooling2D(keepdims=True)(x)
        max_pool = keras.layers.GlobalMaxPooling2D(keepdims=True)(x)
        avg_out = self.channel_dense_2(self.channel_dense_1(avg_pool))
        max_out = self.channel_dense_2(self.channel_dense_1(max_pool))
        channel_attn = keras.activations.sigmoid(avg_out + max_out)
        x = x * channel_attn

        avg_pool_s = keras.ops.mean(x, axis=-1, keepdims=True)
        max_pool_s = keras.ops.max(x, axis=-1, keepdims=True)
        concat = keras.layers.Concatenate(axis=-1)([avg_pool_s, max_pool_s])
        spatial_attn = keras.activations.sigmoid(self.spatial_conv(concat))
        return x * spatial_attn

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"reduction_ratio": self.reduction_ratio, "spatial_kernel": self.spatial_kernel})
        return cfg


model = keras.models.load_model(
    MODEL_PATH, custom_objects={"CBAMBottleneck": CBAMBottleneck}, compile=False)

print("Model loaded:", model.name)
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Total layers:", len(model.layers))

# quick weight sanity check -- confirms this isn't an empty/randomly-initialized model
_w = model.get_layer("bottleneck_cbam").channel_dense_1.get_weights()[0]
print("bottleneck_cbam.channel_dense_1 kernel mean/std:", _w.mean(), _w.std(),
      "-- should NOT be exactly 0/0 (that would mean weights didn't load)")


In [ ]:
# ============================================================
# 3. SANITY CHECK -- run one real image through the model and look at it
# ============================================================
sample_files = sorted(IMAGES_DIR.glob("*.jpg"))
assert len(sample_files) > 0, f"No .jpg files found in {IMAGES_DIR} -- check the path in Section 1."
print(f"Found {len(sample_files)} images in {IMAGES_DIR}")

def load_and_preprocess(path, img_size=IMG_SIZE):
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        return None, None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    orig_hw = img_rgb.shape[:2]
    resized = cv2.resize(img_rgb, (img_size, img_size), interpolation=cv2.INTER_AREA)
    x = resized.astype("float32")
    if PREPROCESS_RESCALE is not None:
        x = x * PREPROCESS_RESCALE
    if PREPROCESS_MEAN is not None and PREPROCESS_STD is not None:
        x = (x - np.array(PREPROCESS_MEAN, dtype="float32")) / np.array(PREPROCESS_STD, dtype="float32")
    return x, orig_hw

sample_path = sample_files[0]
x, orig_hw = load_and_preprocess(sample_path)
pred = model.predict(x[None, ...], verbose=0)[0]
if pred.ndim == 3 and pred.shape[-1] == 1:
    pred = pred[..., 0]

print("Sample file       :", sample_path.name)
print("Original size (HW):", orig_hw)
print("Model input shape :", x.shape)
print("Prediction shape  :", pred.shape, " dtype:", pred.dtype)
print("Prediction min/max/mean:", pred.min(), pred.max(), pred.mean())
print("Fraction of pixels above threshold:", (pred > MASK_THRESHOLD).mean())

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(cv2.resize(cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE)))
axes[0].set_title("Input (resized)")
axes[1].imshow(pred, cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Raw probability map")
axes[2].imshow(pred > MASK_THRESHOLD, cmap="gray")
axes[2].set_title(f"Binary mask @ {MASK_THRESHOLD}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

print("\\nIf this mask looks like noise/empty/all-white, PREPROCESS_* in Section 1 is")
print("probably wrong -- fix that before running the full batch in Section 6.")


In [ ]:
# ============================================================
# 4. (OPTIONAL) COVERAGE CHECK AGAINST Imagewise_Data.csv
# ============================================================
if IMAGEWISE_CSV is not None and IMAGEWISE_CSV.exists():
    df = pd.read_csv(IMAGEWISE_CSV)
    csv_names = set(df["Image Name"].astype(str))
    disk_names = set(p.stem for p in sample_files)
    print("Images in CSV         :", len(csv_names))
    print("Images on disk        :", len(disk_names))
    print("In CSV but not on disk:", len(csv_names - disk_names), sorted(csv_names - disk_names)[:10])
    print("On disk but not in CSV:", len(disk_names - csv_names), sorted(disk_names - csv_names)[:10])
else:
    print("No CSV given (or not found) -- skipping coverage check (not required for inference).")


In [ ]:
# ============================================================
# 5. PER-CATEGORY COMPARISON FIGURES (for your PPT)
#    Picks a few sample images from each class -- Healthy, Benign, OPMD, OCA
#    (Oral Cancer) -- and plots original vs. the model's segmentation side by
#    side. Saves one PNG per class plus one combined overview PNG, all at
#    print-friendly resolution. Only needs Sections 1-2 to have run (model
#    loaded) -- does NOT need the full 3,000-image batch in Section 6.
# ============================================================
SAMPLES_PER_CLASS = 3     # how many example images to show per category
CLASS_ORDER = ["Healthy", "Benign", "OPMD", "OCA"]   # display order
FIGURES_DIR = OUTPUT_DIR.parent / "class_comparison_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGEWISE_CSV is not None and IMAGEWISE_CSV.exists(), (
    f"{IMAGEWISE_CSV} not found -- needed here to know which image belongs to which class. "
    "Re-run Section 1 and give a valid path to Imagewise_Data.csv."
)
cls_df = pd.read_csv(IMAGEWISE_CSV)
cls_df["Image Name"] = cls_df["Image Name"].astype(str)

missing_classes = [c for c in CLASS_ORDER if c not in set(cls_df["Category"])]
if missing_classes:
    print("Warning -- these classes aren't in the CSV's Category column:", missing_classes,
          " Found:", sorted(cls_df["Category"].unique()))

def predict_mask(img_path):
    x, orig_hw = load_and_preprocess(img_path)
    if x is None:
        return None, None, None, None
    pred = model.predict(x[None, ...], verbose=0)[0]
    if pred.ndim == 3 and pred.shape[-1] == 1:
        pred = pred[..., 0]
    mask = (pred > MASK_THRESHOLD).astype("uint8") * 255
    mask_full = cv2.resize(mask, (orig_hw[1], orig_hw[0]), interpolation=cv2.INTER_NEAREST)
    img_rgb_full = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    overlay = img_rgb_full.copy()
    overlay[mask_full > 0] = (0.5 * overlay[mask_full > 0] + 0.5 * np.array([255, 0, 0])).astype("uint8")
    segmented = img_rgb_full.copy()
    segmented[mask_full == 0] = 0   # the actual segmented oral cavity -- background blacked out
    return img_rgb_full, mask_full, overlay, segmented

class_samples = {}
for cls in CLASS_ORDER:
    names = cls_df.loc[cls_df["Category"] == cls, "Image Name"].tolist()
    found = []
    for name in names:
        p = IMAGES_DIR / f"{name}.jpg"
        if p.exists():
            found.append(p)
        if len(found) >= SAMPLES_PER_CLASS:
            break
    class_samples[cls] = found
    print(f"{cls}: using {len(found)} sample image(s) -> {[p.name for p in found]}")

# --- one figure per class: rows = samples, cols = [original, mask, SEGMENTED, overlay] ---
per_class_paths = {}
for cls, paths in class_samples.items():
    if not paths:
        print(f"Skipping {cls} -- no matching image files found on disk.")
        continue
    fig, axes = plt.subplots(len(paths), 4, figsize=(14, 3.6 * len(paths)))
    if len(paths) == 1:
        axes = axes[None, :]
    fig.suptitle(f"{cls} -- original vs. model segmentation", fontsize=15, y=1.02)
    for row, p in enumerate(paths):
        img_rgb, mask_full, overlay, segmented = predict_mask(p)
        axes[row, 0].imshow(img_rgb); axes[row, 0].set_title(p.stem, fontsize=10)
        axes[row, 1].imshow(mask_full, cmap="gray"); axes[row, 1].set_title("Predicted mask", fontsize=10)
        axes[row, 2].imshow(segmented); axes[row, 2].set_title("Segmented oral cavity", fontsize=10)
        axes[row, 3].imshow(overlay); axes[row, 3].set_title("Overlay (QA only)", fontsize=10)
        for c in range(4): axes[row, c].axis("off")
    plt.tight_layout()
    out_fig = FIGURES_DIR / f"{cls}_original_vs_segmented.png"
    fig.savefig(out_fig, dpi=200, bbox_inches="tight")
    per_class_paths[cls] = out_fig
    plt.show()
    print("Saved:", out_fig)

# --- one combined overview figure: rows = classes, cols = [original, SEGMENTED] for sample #1 ---
present_classes = [c for c in CLASS_ORDER if class_samples.get(c)]
fig, axes = plt.subplots(len(present_classes), 2, figsize=(7, 3.6 * len(present_classes)))
if len(present_classes) == 1:
    axes = axes[None, :]
fig.suptitle("One example per class -- original vs. segmentation", fontsize=15, y=1.01)
for row, cls in enumerate(present_classes):
    p = class_samples[cls][0]
    img_rgb, mask_full, overlay, segmented = predict_mask(p)
    axes[row, 0].imshow(img_rgb); axes[row, 0].set_ylabel(cls, fontsize=12, rotation=0, labelpad=40, va="center")
    axes[row, 0].set_title("Original" if row == 0 else "", fontsize=11)
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    axes[row, 1].imshow(segmented)
    axes[row, 1].set_title("Segmented" if row == 0 else "", fontsize=11)
    axes[row, 1].axis("off")
plt.tight_layout()
combined_fig_path = FIGURES_DIR / "all_classes_overview.png"
fig.savefig(combined_fig_path, dpi=200, bbox_inches="tight")
plt.show()

print()
print("Figures for your PPT saved to:", FIGURES_DIR)
for cls, path in per_class_paths.items():
    print(" ", cls, "->", path)
print(" ", "Combined overview ->", combined_fig_path)


In [ ]:
# ============================================================
# 6. FULL-DATASET INFERENCE
#    Runs every image in IMAGES_DIR through the model and, for each one
#    (same base filename, X.jpg -> X.png), saves:
#      - a binary mask                              (OUTPUT_DIR)
#      - the ACTUAL SEGMENTED ORAL CAVITY IMAGE      (SEGMENTED_DIR)
#        -- original photo, full size, everything outside the segmented
#           region blacked out. This is the one you want.
#      - the same segmented image cropped tight to its bounding box
#                                                     (SEGMENTED_CROP_DIR)
#      - optionally a red overlay for visual QA      (OVERLAY_DIR)
#    Originals are only ever opened for reading (cv2.imread) -- nothing in
#    IMAGES_DIR or its parent folders is written to, moved, or deleted, ever.
# ============================================================
import time

all_files = sorted(IMAGES_DIR.glob("*.jpg")) + sorted(IMAGES_DIR.glob("*.jpeg")) + sorted(IMAGES_DIR.glob("*.png"))
print(f"Segmenting {len(all_files)} images...")

failed = []
log_rows = []
t0 = time.time()

for batch_start in range(0, len(all_files), BATCH_SIZE):
    batch_paths = all_files[batch_start: batch_start + BATCH_SIZE]
    batch_x, batch_meta = [], []
    for p in batch_paths:
        x, orig_hw = load_and_preprocess(p)
        if x is None:
            failed.append((p.name, "cv2.imread returned None (corrupt/unreadable file)"))
            continue
        batch_x.append(x)
        batch_meta.append((p, orig_hw))

    if not batch_x:
        continue

    batch_x = np.stack(batch_x, axis=0)
    preds = model.predict(batch_x, verbose=0)
    if preds.ndim == 4 and preds.shape[-1] == 1:
        preds = preds[..., 0]

    for (p, orig_hw), pred in zip(batch_meta, preds):
        mask = (pred > MASK_THRESHOLD).astype("uint8") * 255
        mask_full = cv2.resize(mask, (orig_hw[1], orig_hw[0]), interpolation=cv2.INTER_NEAREST)

        out_path = OUTPUT_DIR / f"{p.stem}.png"
        cv2.imwrite(str(out_path), mask_full)

        lesion_frac = float((mask_full > 0).mean())
        has_region = mask_full.max() > 0

        # --- THE ACTUAL SEGMENTED IMAGE: original pixels inside the mask, black outside ---
        img_bgr_full = cv2.imread(str(p))
        segmented_full = img_bgr_full.copy()
        segmented_full[mask_full == 0] = 0
        seg_path = SEGMENTED_DIR / f"{p.stem}.png"
        cv2.imwrite(str(seg_path), segmented_full)

        # --- same thing, cropped tight to the segmented region's bounding box ---
        crop_path = None
        if has_region:
            ys, xs = np.where(mask_full > 0)
            y0, y1 = ys.min(), ys.max() + 1
            x0, x1 = xs.min(), xs.max() + 1
            segmented_cropped = segmented_full[y0:y1, x0:x1]
            crop_path = SEGMENTED_CROP_DIR / f"{p.stem}.png"
            cv2.imwrite(str(crop_path), segmented_cropped)

        log_rows.append({"image_name": p.stem, "source_file": p.name,
                          "orig_height": orig_hw[0], "orig_width": orig_hw[1],
                          "lesion_pixel_fraction": lesion_frac,
                          "mask_path": str(out_path),
                          "segmented_path": str(seg_path),
                          "segmented_cropped_path": str(crop_path) if crop_path else ""})

        if SAVE_OVERLAYS:
            img_rgb_full = cv2.cvtColor(img_bgr_full, cv2.COLOR_BGR2RGB)
            overlay = img_rgb_full.copy()
            overlay[mask_full > 0] = (0.5 * overlay[mask_full > 0] + 0.5 * np.array([255, 0, 0])).astype("uint8")
            cv2.imwrite(str(OVERLAY_DIR / f"{p.stem}.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

        if SAVE_PROB_MAPS:
            np.save(OUTPUT_DIR / f"{p.stem}_prob.npy", pred.astype("float32"))

    if (batch_start // BATCH_SIZE) % 10 == 0:
        elapsed = time.time() - t0
        done = batch_start + len(batch_x)
        print(f"  {done}/{len(all_files)} done  ({elapsed:.0f}s elapsed)")

print(f"\\nDone in {time.time()-t0:.0f}s. Masks written: {len(log_rows)}. Failed: {len(failed)}.")
if failed:
    print("Failed files:", failed[:20])


In [ ]:
# ============================================================
# 7. SAVE LOG + FINAL AUDIT
# ============================================================
log_df = pd.DataFrame(log_rows)
log_csv_path = OUTPUT_DIR.parent / "segmentation_log.csv"
log_df.to_csv(log_csv_path, index=False)

print("Segmentation log saved to:", log_csv_path)
print()
print("Total source images found :", len(all_files))
print("Masks successfully written:", len(log_df))
print("Failed / unreadable       :", len(failed))
print()
print("Lesion-pixel-fraction distribution (sanity check -- should not be all 0 or all 1):")
print(log_df["lesion_pixel_fraction"].describe())
print()
n_empty = int((log_df["lesion_pixel_fraction"] == 0).sum())
n_full  = int((log_df["lesion_pixel_fraction"] >= 0.99).sum())
print(f"Images with an entirely EMPTY predicted mask : {n_empty} ({n_empty/len(log_df):.1%})")
print(f"Images with an entirely FULL  predicted mask : {n_full} ({n_full/len(log_df):.1%})")
print("A very high share of empty or full masks usually means PREPROCESS_* in Section 1 is")
print("off -- go back and re-check Section 3's sanity-check image.")
print()
print("Output locations:")
print("  Binary masks        :", OUTPUT_DIR, f"({len(list(OUTPUT_DIR.glob('*.png')))} files)")
print("  Segmented images    :", SEGMENTED_DIR, f"({len(list(SEGMENTED_DIR.glob('*.png')))} files)  <-- this is what you want")
print("  Segmented (cropped) :", SEGMENTED_CROP_DIR, f"({len(list(SEGMENTED_CROP_DIR.glob('*.png')))} files)")
if SAVE_OVERLAYS:
    print("  Overlays            :", OVERLAY_DIR, f"({len(list(OVERLAY_DIR.glob('*.png')))} files)")
print("  Log CSV             :", log_csv_path)
print()
print("Your original Images/ and Categories/ folders were only ever read from --")
print("nothing there was modified, moved, or deleted.")


## Notes

- **The actual segmented oral cavity images** (what you asked for -- original
  photo, everything outside the segmented region blacked out) are in
  `segmented_oral_cavity/` (full size) and `segmented_oral_cavity_cropped/`
  (cropped tight to the region), both keyed to the source filename
  (`R-01-01.jpg` -> `R-01-01.png`). `predicted_masks/` (plain black/white) and
  `predicted_overlays/` (red highlight on the original, for visual QA only)
  are separate, secondary outputs -- not what you want if you're after the
  segmented image itself.
- **For the PPT**: run Sections 1, 2, and 5 only -- Section 5 makes one PNG per
  class (`Healthy_original_vs_segmented.png`, etc., now with 4 columns:
  original / mask / segmented / overlay) plus `all_classes_overview.png`
  (original vs. segmented, one row per class), all saved to
  `class_comparison_figures/` at print resolution (200 dpi). You do **not**
  need to run the full 3,000-image batch (Section 6) just to get these.
- **For the full segmented dataset**: run Sections 1, 2, then 6-7.
- **If Section 3's sanity-check mask looks wrong** (empty, solid, or noisy):
  check `PREPROCESS_RESCALE` / `PREPROCESS_MEAN` / `PREPROCESS_STD` in Section 1
  first -- a mismatched normalization is the most common cause of a model that
  loads fine but predicts garbage. If the mask is wrong, the segmented image
  built from it will be wrong too.
- **Disk usage**: 3,000 images written 2-3x over (masks + segmented + cropped
  + optional overlays) can add up -- turn `SAVE_OVERLAYS`/`SAVE_PROB_MAPS` off
  in Section 1 if you're low on disk space.
